# 📊 Batch Size Benchmark — Exploratory Analysis

Notebook นี้ใช้สำรวจผล benchmark ของ dlt pipeline แบบ **local 100%** (อ่านจากไฟล์ `benchmark.duckdb` ไฟล์เดียว ไม่ต้องต่อ server อะไรเลย)

Notebook ทำได้ **2 โหมด** (เลือกได้ในเซลล์ที่ 2):

1. **ดึง log เดิม** — อ่านตาราง `benchmark_runs` ที่ pipeline เคยเขียนไว้แล้ว มาวาดกราฟเลย
2. **Run จริง (mini-sweep)** — สั่ง `run_pipeline()` วนหลาย batch size ด้วยข้อมูลชุดเล็ก เพื่อให้ได้ข้อมูลหลายจุดมาวาดเป็นเส้นโค้ง (ทดสอบ hypothesis H1/H3 ได้จริง)

**เป้าหมาย:** ดูว่า *"batch ใหญ่ขึ้น = เร็วขึ้นเสมอ"* จริงไหม → คาดว่ากราฟจะเป็นรูป **inverted-U** ไม่ใช่เส้นตรง

## 0. Setup — import และตั้งค่า path

เซลล์นี้ทำ 3 อย่าง:
- ย้าย working directory ไปที่ราก project (เพื่อให้ path ใน `.env` ที่เป็น relative ชี้ถูกไฟล์)
- เพิ่ม project root เข้า `sys.path` เพื่อ `import` โมดูล `dlt_pipelines` ได้
- โหลด `.env` และอ่าน path ของ DuckDB

In [1]:
import os
import sys
import logging
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

# notebook อยู่ใน notebooks/ -> ราก project คือ parent
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)                 # ทำให้ path relative ใน .env ใช้ได้
sys.path.insert(0, str(ROOT))  # ทำให้ import dlt_pipelines ได้

load_dotenv(ROOT / ".env")
logging.basicConfig(level=logging.WARNING)  # ลด noise ของ log dlt ใน notebook

DUCKDB_PATH = Path(os.getenv("DUCKDB_PATH", "data/duckdb/benchmark.duckdb")).resolve()
print("project root :", ROOT)
print("duckdb file  :", DUCKDB_PATH, "(exists:", DUCKDB_PATH.exists(), ")")

project root : E:\01\dowload\project Data Architecture\batch_benchmark
duckdb file  : E:\01\dowload\project Data Architecture\batch_benchmark\data\duckdb\benchmark.duckdb (exists: True )


## 1. ดึง log เดิมจาก DuckDB (ตาราง `benchmark_runs`)

ทุกครั้งที่ `run_pipeline()` ทำงานเสร็จ มันจะ **append 1 แถว** ลงตาราง `benchmark_runs`
(run_id, batch_size, row_width, throughput ฯลฯ) เซลล์ล่างนี้แค่อ่านตารางนั้นเข้ามาเป็น DataFrame — **ยังไม่ได้ run อะไรใหม่**

In [2]:
def load_runs() -> pd.DataFrame:
    """อ่านตาราง benchmark_runs จาก DuckDB (โหมด read-only). คืน DataFrame ว่างถ้ายังไม่มีตาราง"""
    if not DUCKDB_PATH.exists():
        return pd.DataFrame()
    con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
    try:
        return con.execute("SELECT * FROM benchmark_runs ORDER BY start_time").df()
    except duckdb.Error:
        return pd.DataFrame()
    finally:
        con.close()


runs = load_runs()
print(f"พบ {len(runs)} run ใน log เดิม")
runs

พบ 1 run ใน log เดิม


,run_id,batch_size,row_width,complexity,total_rows,rows_loaded,start_time,end_time,duration_sec,throughput_rows_per_sec,repetition,dataset_name
0,6d7f1e89ac694a2eaabf69d3d30ad054,10000,medium,simple,50000,50000,2026-05-16 01:24:22.680757,2026-05-16 01:24:49.020119,26.339362,1898.299587,1,bench_6d7f1e89


## 2. (ตัวเลือก) Run mini-sweep จริง

ถ้า log เดิมมีแค่ run เดียว เราจะวาดเส้นโค้งไม่ได้ → เซลล์นี้สั่ง pipeline ทำงานจริง โดยวน batch size หลายค่า ด้วยข้อมูล **ชุดเล็ก** (เร็ว ~1 นาที)

- ตั้ง `RUN_LIVE = True` เพื่อ run จริง / `False` เพื่อใช้แต่ log เดิม
- `MINI_TOTAL_ROWS` เล็กไว้ก่อน (ปรับขึ้นได้ถ้าอยากให้ผลนิ่งขึ้น)
- ผลแต่ละ run จะถูก append ลง `benchmark_runs` อัตโนมัติ (สะสมเรื่อย ๆ)

In [3]:
from dlt_pipelines.pipeline import run_pipeline

RUN_LIVE = True                                  # << เปลี่ยนเป็น False ถ้าไม่อยาก run ใหม่
MINI_TOTAL_ROWS = 12_000                         # ข้อมูลชุดเล็กเพื่อความเร็ว
MINI_BATCH_SIZES = [100, 1_000, 5_000, 10_000, 50_000]
MINI_ROW_WIDTH = "medium"                         # ลองเปลี่ยนเป็น narrow/wide เพื่อทดสอบ H4

if RUN_LIVE:
    for bs in MINI_BATCH_SIZES:
        r = run_pipeline(
            batch_size=bs,
            total_rows=MINI_TOTAL_ROWS,
            row_width=MINI_ROW_WIDTH,
            complexity="simple",
            repetition=1,
        )
        print(f"batch_size={bs:>7,} -> {r.throughput_rows_per_sec:8.1f} rows/sec  ({r.duration_sec:.2f}s)")
else:
    print("ข้าม mini-sweep — ใช้ log เดิมอย่างเดียว")

batch_size=    100 ->   1242.7 rows/sec  (9.66s)


batch_size=  1,000 ->   1617.5 rows/sec  (7.42s)


batch_size=  5,000 ->   1967.9 rows/sec  (6.10s)


batch_size= 10,000 ->   1591.7 rows/sec  (7.54s)


batch_size= 50,000 ->   1580.6 rows/sec  (7.59s)


### โหลดข้อมูลใหม่หลัง sweep
อ่าน `benchmark_runs` อีกรอบให้รวมผลที่เพิ่ง run เข้ามาด้วย

In [4]:
runs = load_runs()
print(f"รวมทั้งหมด {len(runs)} run")
runs[["batch_size", "row_width", "total_rows", "rows_loaded", "duration_sec", "throughput_rows_per_sec"]]

รวมทั้งหมด 6 run


,batch_size,row_width,total_rows,rows_loaded,duration_sec,throughput_rows_per_sec
0,10000,medium,50000,50000,26.339362,1898.299587
1,100,medium,12000,12000,9.656282,1242.714328
2,1000,medium,12000,12000,7.419052,1617.457325
3,5000,medium,12000,12000,6.097985,1967.863155
4,10000,medium,12000,12000,7.539025,1591.717762
5,50000,medium,12000,12000,7.592248,1580.559539


## 3. กราฟที่ 1 — Throughput Curve (ทดสอบ H1)

**H1 บอกว่า:** throughput เพิ่มขึ้นตาม batch size แบบเส้นตรง (monotonic)

เราหาค่า **median throughput** ต่อ batch size (กันค่า noise) แล้ววาดเส้น โดยแกน X เป็น **log scale**
ถ้ากราฟ **โค้งลงปลาย** (inverted-U) = H1 ถูกหักล้าง และมี "sweet spot" จริง

In [5]:
assert not runs.empty, "ยังไม่มีข้อมูล — ตั้ง RUN_LIVE=True แล้ว run เซลล์ที่ 2 ใหม่"

curve = (
    runs.groupby(["batch_size", "row_width"], as_index=False)
    .agg(throughput=("throughput_rows_per_sec", "median"),
         duration=("duration_sec", "median"))
    .sort_values("batch_size")
)

fig = px.line(
    curve, x="batch_size", y="throughput", color="row_width", markers=True,
    log_x=True, title="Throughput vs Batch Size (median)",
    labels={"batch_size": "Batch Size (rows, log scale)", "throughput": "Throughput (rows/sec)"},
)

# ทำเครื่องหมาย sweet spot = จุด throughput สูงสุด
best = curve.loc[curve["throughput"].idxmax()]
fig.add_annotation(x=best["batch_size"], y=best["throughput"],
                   text=f"sweet spot ≈ {int(best['batch_size']):,}",
                   showarrow=True, arrowhead=2, ay=-40)
fig.update_layout(template="plotly_white")
fig

## 4. กราฟที่ 2 — Duration vs Batch Size

ดูเวลาที่ใช้ทั้ง run ถ้า batch เล็กมาก (เช่น 100) จะเสีย overhead ต่อ batch เยอะ → ช้า;
ถ้า batch ใหญ่เกินไปก็อาจช้าเพราะ memory pressure → ควรเห็นเป็นรูป **U**

In [6]:
fig = px.line(
    curve, x="batch_size", y="duration", color="row_width", markers=True,
    log_x=True, title="Run Duration vs Batch Size (median)",
    labels={"batch_size": "Batch Size (rows, log scale)", "duration": "Duration (sec)"},
)
fig.update_layout(template="plotly_white")
fig

## 5. กราฟที่ 3 — เทียบ row_width (ทดสอบ H4)

**H4 บอกว่า:** batch size ที่ดีที่สุดขึ้นกับ "ความกว้างของแถว" (จำนวน column)

ถ้าใน log มีหลาย `row_width` (narrow/medium/wide) bar chart นี้จะเทียบ throughput สูงสุดของแต่ละแบบให้เห็น
(ถ้ามีแค่แบบเดียว ให้ย้อนไปเซลล์ที่ 2 เปลี่ยน `MINI_ROW_WIDTH` แล้ว run เพิ่ม)

In [7]:
widths = curve["row_width"].nunique()
if widths >= 2:
    peak = curve.loc[curve.groupby("row_width")["throughput"].idxmax()]
    fig = px.bar(
        peak, x="row_width", y="throughput", color="row_width", text="batch_size",
        title="Peak Throughput ต่อ row_width (text = batch size ที่ดีที่สุด)",
        labels={"throughput": "Peak Throughput (rows/sec)", "row_width": "Row Width"},
    )
    fig.update_layout(template="plotly_white")
    display(fig)
else:
    print(f"มี row_width แค่ {widths} แบบ — ข้ามกราฟนี้ (เปลี่ยน MINI_ROW_WIDTH แล้ว run เพิ่มเพื่อทดสอบ H4)")

มี row_width แค่ 1 แบบ — ข้ามกราฟนี้ (เปลี่ยน MINI_ROW_WIDTH แล้ว run เพิ่มเพื่อทดสอบ H4)


## 6. หา Sweet Spot + ตารางสรุป

สรุปตัวเลขให้อ่านง่าย: batch size ไหนให้ throughput สูงสุด และเร็วกว่า batch เล็กสุด/ใหญ่สุดกี่ %

In [8]:
summary = curve.copy()
summary["throughput"] = summary["throughput"].round(1)
summary["duration"] = summary["duration"].round(2)

best = summary.loc[summary["throughput"].idxmax()]
worst_small = summary.iloc[0]
worst_big = summary.iloc[-1]

print("=== สรุปผล (median ต่อ batch size) ===")
print(summary.to_string(index=False))
print()
print(f"🏆 Sweet spot: batch_size = {int(best['batch_size']):,} "
      f"({best['row_width']}) -> {best['throughput']:,} rows/sec")
if best["throughput"] and worst_small["throughput"]:
    gain = (best["throughput"] / worst_small["throughput"] - 1) * 100
    print(f"   เร็วกว่า batch เล็กสุด ({int(worst_small['batch_size']):,}) ~ {gain:.0f}%")
if best["batch_size"] != worst_big["batch_size"] and worst_big["throughput"]:
    drop = (1 - worst_big["throughput"] / best["throughput"]) * 100
    print(f"   batch ใหญ่สุด ({int(worst_big['batch_size']):,}) กลับ "
          f"{'ช้าลง' if drop > 0 else 'เร็วขึ้น'} ~ {abs(drop):.0f}% จาก sweet spot")

=== สรุปผล (median ต่อ batch size) ===
 batch_size row_width  throughput  duration
        100    medium      1242.7      9.66
       1000    medium      1617.5      7.42
       5000    medium      1967.9      6.10
      10000    medium      1745.0     16.94
      50000    medium      1580.6      7.59

🏆 Sweet spot: batch_size = 5,000 (medium) -> 1,967.9 rows/sec
   เร็วกว่า batch เล็กสุด (100) ~ 58%
   batch ใหญ่สุด (50,000) กลับ ช้าลง ~ 20% จาก sweet spot


## 7. สรุปผล (ตอบ Hypothesis)

เติมข้อสรุปหลังดูกราฟด้านบน:

| Hypothesis | คาดไว้ | ผลจริง (เติมเอง) |
|---|---|---|
| **H1** throughput เพิ่มแบบเส้นตรงตาม batch | ❌ น่าจะเป็น inverted-U | _…_ |
| **H3** มี sweet spot ช่วง 5K–50K | ✅ | _ดูค่าจากเซลล์ที่ 6_ |
| **H4** optimal batch ขึ้นกับ row_width | ✅ | _ต้อง run หลาย row_width เพิ่ม_ |

**หมายเหตุ:** ข้อมูลชุดเล็ก (12K rows) ใช้ดู *แนวโน้ม* — เลขสัมบูรณ์จะนิ่งขึ้นเมื่อเพิ่ม `MINI_TOTAL_ROWS` หรือใช้ข้อมูลจากการ sweep เต็ม (Dagster) ในขั้นถัดไป